# 12.9 - Production Incidents

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook works through Production Incidents hands-on: See the gray failure uptime is blind to; Classify severity and read the two clocks; Declare the incident and separate the roles; Triage by asking what changed; Mitigate first with the five rollback levers; The two AI-specific runaways.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

This is a one-line reproducibility note - the lesson leans on fixed, hand-written numbers rather than live random draws, so every run of every block prints the same output. There is nothing to install and no imports here; the later cells are pure standard-library Python.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A single comment cell that flags the notebook is deterministic by design - the incident scenarios use seeded, hard-coded values so the numbers you see match the walkthroughs exactly.

**How the code works - step by step**
- The comment records that the lesson uses seeded random values throughout, so outputs are stable across runs.

**In one line:** deterministic-by-design, nothing to install.

**What you'll see:** (no output - environment prep)

## 1 - See the gray failure uptime is blind to

A normal outage returns a 500 and stops; an AI incident keeps returning a clean **HTTP 200 with a confidently wrong answer**. This block runs the same batch of six responses through two lenses - an uptime monitor and an AI health check - to show how a dashboard can be all-green while the assistant is on fire.

In [ ]:
# A traditional bug 500s and stops. An AI incident returns HTTP 200 with valid JSON
# that is confidently WRONG - a "gray failure" the uptime monitor cannot see.
# (each response is HTTP 200; the AI health check inspects quality / PII / cost)
responses = [
    {"id": "r1", "status": 200, "quality": 8.6, "has_pii": False, "cost_usd": 0.011},
    {"id": "r2", "status": 200, "quality": 3.1, "has_pii": False, "cost_usd": 0.012},   # hallucination
    {"id": "r3", "status": 200, "quality": 8.2, "has_pii": True,  "cost_usd": 0.010},   # PII leak
    {"id": "r4", "status": 200, "quality": 8.4, "has_pii": False, "cost_usd": 0.013},
    {"id": "r5", "status": 200, "quality": 8.0, "has_pii": False, "cost_usd": 0.190},   # cost spike
    {"id": "r6", "status": 200, "quality": 2.8, "has_pii": False, "cost_usd": 0.012},   # hallucination
]
def ai_health(r):
    if r["has_pii"]:        return "PII LEAK"
    if r["quality"] < 5.0:  return "HALLUCINATION"
    if r["cost_usd"] > 0.10: return "COST SPIKE"
    return None

up = sum(1 for r in responses if r["status"] == 200)
print("UPTIME MONITOR (is it up / fast / erroring?):")
print("  {}/{} responses returned HTTP 200 -> 0 incidents, dashboard GREEN".format(up, len(responses)))
print()
print("AI HEALTH CHECK (is the answer any good?):")
incidents = [(r["id"], ai_health(r)) for r in responses if ai_health(r)]
for rid, kind in incidents:
    print("  {}: HTTP 200, but {}".format(rid, kind))
print("  -> {} gray failures the uptime monitor never saw".format(len(incidents)))
print()
print("An AI incident does not 500 and stop; it keeps answering, 200 and wrong.")
print("You detect it with quality / cost / safety signals (Lesson 12.8), not uptime.")

# Output:
# UPTIME MONITOR (is it up / fast / erroring?):
#   6/6 responses returned HTTP 200 -> 0 incidents, dashboard GREEN
#
# AI HEALTH CHECK (is the answer any good?):
#   r2: HTTP 200, but HALLUCINATION
#   r3: HTTP 200, but PII LEAK
#   r5: HTTP 200, but COST SPIKE
#   r6: HTTP 200, but HALLUCINATION
#   -> 4 gray failures the uptime monitor never saw
#
# An AI incident does not 500 and stop; it keeps answering, 200 and wrong.
# You detect it with quality / cost / safety signals (Lesson 12.8), not uptime.

**Explanation**

This is a side-by-side detector, not a model call: it hard-codes six responses (all status 200) that carry quality, PII, and cost fields, then contrasts what an uptime monitor sees against what a content-aware health check sees. The whole point is that transport-layer success (200) and answer-layer failure are two different axes, and only the second sensor catches the gray failures.

**How the code works - step by step**
- **`responses`** - a list of six dicts, every one HTTP 200, but with hidden problems: two low-quality (hallucination), one `has_pii`, one with a 0.19 cost spike.
- **`ai_health`** - a rules ladder over one response: PII first, then quality below 5.0, then cost above 0.10; returns the failure kind or `None` if healthy.
- **uptime block** - counts the 200s (6/6) and prints the green-dashboard verdict: zero incidents.
- **AI health block** - runs `ai_health` over every response, collects the ones that flag, and prints each gray failure with its kind.

**In one line:** all six pass the uptime check; four fail the answer check.

**What you'll see:** The uptime monitor reports 6/6 HTTP 200 -> dashboard GREEN, while the AI health check flags 4 gray failures (r2 and r6 HALLUCINATION, r3 PII LEAK, r5 COST SPIKE) that uptime never saw, closing with the reminder that you detect these with quality/cost/safety signals, not uptime.

## 2 - Classify severity and read the two clocks

Not every issue is a 3 AM page, so you classify by impact and respond proportionally - then measure how you did with two numbers: **MTTD** (how long it was broken before anyone knew) and **MTTR** (detect-to-recover). This block maps signals to SEV1-4 with response targets, then walks a silent quality drop across its timeline.

In [ ]:
# Classify by impact so you respond proportionally - then measure the two clocks.
def classify(sig):
    # SEV1: critical (safety/legal/outage/cost runaway)
    if sig.get("pii_leak") or sig.get("safety_violation") or sig.get("total_outage"): return "SEV1"
    if sig.get("cost_per_hour", 0) > 100: return "SEV1"
    # SEV2: major (the "silent killer" - quality/hallucination/big errors)
    if sig.get("error_rate_pct", 0) > 20: return "SEV2"
    if sig.get("quality_drop", 0) >= 2:   return "SEV2"
    if sig.get("hallucination_pct", 0) > 15: return "SEV2"
    if sig.get("cost_per_hour", 0) > 50:  return "SEV2"
    # SEV3: minor
    if sig.get("error_rate_pct", 0) > 5:  return "SEV3"
    if sig.get("quality_drop", 0) >= 1:   return "SEV3"
    return "SEV4"

TARGETS = {"SEV1": "ack 5m / mitigate 30m", "SEV2": "ack 15m / mitigate 2h",
           "SEV3": "ack 1h / mitigate 1d", "SEV4": "ack 1d / mitigate 1wk"}
cases = [
    ("PII in a chatbot reply",        {"pii_leak": True}),
    ("Cost at $80/hour",              {"cost_per_hour": 80}),
    ("Quality dropped 2.5 points",    {"quality_drop": 2.5}),
    ("Error rate 8%",                 {"error_rate_pct": 8}),
    ("Slightly slower responses",     {"latency_p99_ms": 5000}),
]
print("Severity from signals:")
for name, sig in cases:
    sev = classify(sig)
    print("  {:<28} -> {}  ({})".format(name, sev, TARGETS[sev]))
print()
# The two clocks. MTTD = how long it was broken before anyone knew. MTTR = detect -> resolve.
# Timeline in minutes from when the incident BEGAN (a silent SEV2 quality drop):
began, detected, resolved = 0, 240, 275
mttd = detected - began
mttr = resolved - detected
print("The clock on a silent SEV2 (quality quietly dropped):")
print("  began t={}m   detected t={}m   resolved t={}m".format(began, detected, resolved))
print("  MTTD (broken before anyone knew) = {}m".format(mttd))
print("  MTTR (detect -> recover)         = {}m".format(mttr))
print("  The kill was the {}m MTTD, not the {}m fix. A short MTTD predicts a short MTTR:".format(mttd, mttr))
print("  detection is the highest-leverage number to improve.")

# Output:
# Severity from signals:
#   PII in a chatbot reply       -> SEV1  (ack 5m / mitigate 30m)
#   Cost at $80/hour             -> SEV2  (ack 15m / mitigate 2h)
#   Quality dropped 2.5 points   -> SEV2  (ack 15m / mitigate 2h)
#   Error rate 8%                -> SEV3  (ack 1h / mitigate 1d)
#   Slightly slower responses    -> SEV4  (ack 1d / mitigate 1wk)
#
# The clock on a silent SEV2 (quality quietly dropped):
#   began t=0m   detected t=240m   resolved t=275m
#   MTTD (broken before anyone knew) = 240m
#   MTTR (detect -> recover)         = 35m
#   The kill was the 240m MTTD, not the 35m fix. A short MTTD predicts a short MTTR:
#   detection is the highest-leverage number to improve.

**Explanation**

Two things in one cell: a severity classifier that turns raw signals into a SEV level with an ack/mitigate target, and a tiny timeline calculation that separates the undetected window from the fix window. It shows that for an AI app the long, silent MTTD - not the fix - is usually where the damage lives.

**How the code works - step by step**
- **`classify`** - a priority ladder: PII/safety/outage or cost over $100/hr is SEV1; big error rate, a 2-point quality drop, a hallucination spike, or cost over $50/hr is SEV2; smaller error/quality is SEV3; everything else SEV4.
- **`TARGETS`** - the ack-and-mitigate window per severity, so urgency is never debated live.
- **`cases` loop** - runs five example signals through `classify` and prints each severity with its target.
- **the clock** - sets `began=0, detected=240, resolved=275`, computes `mttd = detected - began` and `mttr = resolved - detected`, and prints both against the timeline.

**In one line:** score the signal into a SEV bucket, then split the timeline into MTTD (silent) and MTTR (fix).

**What you'll see:** A severity table (PII -> SEV1; $80/hr and a 2.5-point quality drop -> SEV2; 8% errors -> SEV3; slight slowdown -> SEV4), then the clock on a silent SEV2 showing MTTD = 240m versus MTTR = 35m, driving home that the 240m detection gap - not the 35m fix - was the real kill.

## 3 - Declare the incident and separate the roles

A declared SEV1 fails when ten engineers type conflicting commands at once. The fix is three deliberately separated roles: an **Incident Commander** who coordinates and decides (but does not fix), an **Operations lead** who executes, and a **Communications lead** who owns the status page. This block routes each action to its owner and rejects the classic anti-pattern.

In [ ]:
# In a SEV1 you do not want ten people typing gcloud at once. Assign three SEPARATED roles.
ROLES = {
    "Incident Commander": "coordinates, assigns roles, decides - does NOT do hands-on fixes",
    "Operations Lead":    "drives the diagnosis and runs the mitigation commands",
    "Communications Lead":"owns the status page and stakeholder / customer updates",
}
def owner(action):
    a = action.lower()
    if "decide" in a or "coordinate" in a or "assign" in a: return "Incident Commander"
    if "run " in a or "read the traces" in a or "roll back" in a or "execute" in a: return "Operations Lead"
    if "status" in a or "notify" in a or "customer" in a: return "Communications Lead"
    return "Operations Lead"

print("Roles in a declared SEV1:")
for r, duty in ROLES.items():
    print("  {:<20} {}".format(r, duty))
print()
actions = [
    "Decide whether to roll back",
    "Run the traffic rollback command",
    "Read the traces to find the cause",
    "Post the status-page update",
    "Notify the enterprise customers",
]
print("Who owns each action:")
for a in actions:
    print("  {:<38} -> {}".format(a, owner(a)))
print()
# The anti-pattern: the IC going head-down to type the fix.
print("Anti-pattern check:")
print("  'The Incident Commander personally types the gcloud fix'")
print("  -> REJECTED: the IC coordinates and decides; the Ops Lead executes.")
print("  An IC head-down in a terminal loses the big picture the incident needs.")

# Output:
# Roles in a declared SEV1:
#   Incident Commander   coordinates, assigns roles, decides - does NOT do hands-on fixes
#   Operations Lead      drives the diagnosis and runs the mitigation commands
#   Communications Lead  owns the status page and stakeholder / customer updates
#
# Who owns each action:
#   Decide whether to roll back            -> Incident Commander
#   Run the traffic rollback command       -> Operations Lead
#   Read the traces to find the cause      -> Operations Lead
#   Post the status-page update            -> Communications Lead
#   Notify the enterprise customers        -> Communications Lead
#
# Anti-pattern check:
#   'The Incident Commander personally types the gcloud fix'
#   -> REJECTED: the IC coordinates and decides; the Ops Lead executes.
#   An IC head-down in a terminal loses the big picture the incident needs.

**Explanation**

A router that assigns responsibilities, not a model call: it defines the three roles, then keyword-matches each incident action to exactly one owner, and finally hard-codes the anti-pattern check that keeps the IC out of the terminal. The lesson is separation of duties - one person holding the big picture while another runs the commands.

**How the code works - step by step**
- **`ROLES`** - the three roles and their one-line duties, with the IC explicitly *not* doing hands-on fixes.
- **`owner`** - keyword rules over an action string: decide/coordinate/assign -> IC; run/read the traces/roll back/execute -> Ops; status/notify/customer -> Comms; default Ops.
- **actions loop** - routes five real actions (decide to roll back, run the rollback, read traces, post status, notify customers) to their owner.
- **anti-pattern block** - prints the rejected move: the IC personally typing the fix, and why an IC head-down in a terminal loses the big picture.

**In one line:** one owner per action, and never the IC on the keyboard.

**What you'll see:** The three roles with duties, then each action mapped to its owner (decide -> IC; run rollback and read traces -> Ops; post status and notify customers -> Comms), and an anti-pattern check that REJECTS the IC typing the gcloud fix - the IC coordinates, the Ops lead executes.

## 4 - Triage by asking what changed

With someone in charge, the Ops lead has to localize the fault fast, and the fastest question is **what changed?** An AI app can break at five layers - model, prompt, data, cache, or infra. This block bisects to the culprit by scoring each recent change on two axes: does it explain the symptom, and does its timing match the onset?

In [ ]:
# Triage = bisect to the culprit LAYER by asking "what changed?". Score each recent
# change by whether it explains the symptoms AND whether its timing matches the onset.
onset_min_ago = 20   # quality started dropping ~20 minutes ago
changes = [
    {"layer": "PROMPT", "what": "prompt deploy",       "age_min": 20,   "affects": "quality"},
    {"layer": "MODEL",  "what": "provider version bump","age_min": 4320, "affects": "quality"},
    {"layer": "DATA",   "what": "RAG reindex",          "age_min": 1440, "affects": "quality"},
    {"layer": "INFRA",  "what": "autoscaler config",    "age_min": 30,   "affects": "latency"},
]
symptoms = {"quality": True, "error_rate": False, "cost": False, "latency": False}

def score(c):
    s = 0
    if symptoms.get(c["affects"]):                 s += 2   # explains the symptom
    if abs(c["age_min"] - onset_min_ago) <= 15:    s += 2   # timing matches the onset
    elif c["age_min"] <= 60:                       s += 1
    return s

print("Symptoms: quality DOWN; error rate / cost / latency all FLAT.")
print()
print("Rank recent changes by (explains symptom) + (timing matches onset):")
ranked = sorted(changes, key=score, reverse=True)
for c in ranked:
    print("  {:<6} {:<22} {:>5}m ago   score {}".format(c["layer"], c["what"], c["age_min"], score(c)))
print()
top = ranked[0]
print("Bisect: INFRA ruled out (latency flat), MODEL / DATA changes are too old to be the onset,")
print("so the culprit is the {} ({}, {}m ago) - it explains a quality-only drop AND matches the timing.".format(top["layer"], top["what"], top["age_min"]))
print("Confirm by rolling that one change back (next step) before hunting the exact line.")

# Output:
# Symptoms: quality DOWN; error rate / cost / latency all FLAT.
#
# Rank recent changes by (explains symptom) + (timing matches onset):
#   PROMPT prompt deploy             20m ago   score 4
#   MODEL  provider version bump   4320m ago   score 2
#   DATA   RAG reindex             1440m ago   score 2
#   INFRA  autoscaler config         30m ago   score 2
#
# Bisect: INFRA ruled out (latency flat), MODEL / DATA changes are too old to be the onset,
# so the culprit is the PROMPT (prompt deploy, 20m ago) - it explains a quality-only drop AND matches the timing.
# Confirm by rolling that one change back (next step) before hunting the exact line.

**Explanation**

A scoring bisect, not a model call: it lines up the observed symptoms (quality down, everything else flat) against a changelog and ranks each change by explanatory power plus timing fit. It shows how the 12.8 telemetry lets you rule layers in and out instead of guessing - the culprit is the change that both *could* cause this symptom and started *when* it started.

**How the code works - step by step**
- **`onset_min_ago`** and **`changes`** - the drop began ~20m ago; four candidate changes carry a layer, an age, and which axis they affect.
- **`symptoms`** - the observed picture: quality True, error_rate/cost/latency all False (flat).
- **`score`** - +2 if the change affects an active symptom, +2 if its age is within 15m of onset (else +1 if under 60m), so relevance and timing both count.
- **ranking** - sorts the changes by score descending and prints each with its score; then names the top scorer.

**In one line:** score every recent change on symptom-fit + timing-fit and take the winner.

**What you'll see:** Symptoms print as quality DOWN with everything flat, then the ranked changes - the PROMPT deploy (20m ago) wins at score 4 while the model bump, RAG reindex, and infra change tie at 2 - concluding that the prompt is the culprit (explains a quality-only drop and matches the timing), to be confirmed by rolling it back.

## 5 - Mitigate first with the five rollback levers

The discipline that separates a calm responder from a panicked one: **mitigate first, diagnose second**. This block lays out the five escalating rollback levers - traffic, model, cache, throttle, and the kill switch - each with a blast radius and a time-to-effect, then picks the right one per incident. Every lever restores users before you know the root cause.

In [ ]:
# Mitigate to restore users BEFORE you know the root cause. Five escalating levers.
LEVERS = {
    "traffic":  {"blast": "1 deploy",      "effect": "instant",   "use": "a bad deploy (quality/error regression)"},
    "model":    {"blast": "all requests",  "effect": "seconds",   "use": "the current model hallucinating"},
    "cache":    {"blast": "cached hits",   "effect": "seconds",   "use": "cache serving stale / wrong / PII"},
    "throttle": {"blast": "peak traffic",  "effect": "seconds",   "use": "a cost runaway / retry storm"},
    "kill":     {"blast": "the AI feature","effect": "instant",   "use": "a systematic PII / safety leak"},
}
def pick(incident):
    i = incident.lower()
    if "deploy" in i or "regression" in i:     return "traffic"
    if "hallucinat" in i:                       return "model"
    if "cache" in i or "stale" in i:            return "cache"
    if "cost" in i or "retry" in i or "budget" in i: return "throttle"
    if "pii" in i or "safety" in i:             return "kill"
    return "traffic"

print("The five rollback levers (mitigate first, diagnose second):")
for name, l in LEVERS.items():
    print("  {:<9} blast={:<13} effect={:<8} use when: {}".format(name, l["blast"], l["effect"], l["use"]))
print()
incidents = [
    "New deploy caused a quality regression",
    "Current model is hallucinating",
    "Semantic cache returning stale answers",
    "Cost runaway from a retry storm",
    "Systematic PII leak in outputs",
]
print("Pick the lever per incident:")
for inc in incidents:
    lv = pick(inc)
    print("  {:<42} -> {:<9} ({})".format(inc, lv, LEVERS[lv]["effect"]))
print()
print("Every lever restores users NOW; the root cause can wait for the calm after.")

# Output:
# The five rollback levers (mitigate first, diagnose second):
#   traffic   blast=1 deploy      effect=instant  use when: a bad deploy (quality/error regression)
#   model     blast=all requests  effect=seconds  use when: the current model hallucinating
#   cache     blast=cached hits   effect=seconds  use when: cache serving stale / wrong / PII
#   throttle  blast=peak traffic  effect=seconds  use when: a cost runaway / retry storm
#   kill      blast=the AI feature effect=instant  use when: a systematic PII / safety leak
#
# Pick the lever per incident:
#   New deploy caused a quality regression     -> traffic   (instant)
#   Current model is hallucinating             -> model     (seconds)
#   Semantic cache returning stale answers     -> cache     (seconds)
#   Cost runaway from a retry storm            -> throttle  (seconds)
#   Systematic PII leak in outputs             -> kill      (instant)
#
# Every lever restores users NOW; the root cause can wait for the calm after.

**Explanation**

A lever-picker that encodes the mitigation playbook: it catalogs five levers by blast radius and speed, then keyword-routes each incident to the smallest lever that stops the danger. The theme is graceful degradation - a safe fallback beats a confidently wrong answer, so you flip a lever now and investigate in the calm afterwards.

**How the code works - step by step**
- **`LEVERS`** - the five levers, each with a blast radius, time-to-effect, and the situation it is for (traffic for a bad deploy, model for hallucination, cache for stale hits, throttle for a cost runaway, kill for a PII/safety leak).
- **`pick`** - keyword rules over an incident string: deploy/regression -> traffic; hallucinat -> model; cache/stale -> cache; cost/retry/budget -> throttle; pii/safety -> kill.
- **catalog print** - lists every lever with its properties.
- **incidents loop** - routes five incidents to a lever and prints the lever plus its time-to-effect.

**In one line:** match the incident to the smallest lever that stops the bleeding, root cause later.

**What you'll see:** The five levers printed with blast radius and effect, then each incident mapped to its lever (bad deploy -> traffic/instant, hallucination -> model, stale cache -> cache, cost runaway -> throttle, PII leak -> kill), closing on the rule that every lever restores users now and the root cause can wait.

## 6 - The two AI-specific runaways

Two AI incidents feed on themselves and deserve a dedicated fast path. A **retry storm** amplifies - failures spawn retries that pile more load on a degraded provider and burn the budget - and a **PII leak**'s blast radius includes the cache, so one leaked answer is replayed thousands of times. This block simulates both and shows the circuit breaker flattening the storm.

In [ ]:
# TWO AI-specific runaways. (1) A retry storm amplifies: failures spawn retries, which
# add load, which cause more failures. A circuit breaker + budget cap flattens it.
BASE = 100          # steady request load
FAIL_RATE = 0.5     # half are failing (provider degraded)
RETRIES = 3         # each failure is retried 3x
COST_PER = 0.002    # $ per attempt

def run(breaker):
    load, total_cost = BASE, 0.0
    print("  tick  attempts  failures  retries_added  cost")
    for t in range(4):
        failures = int(load * FAIL_RATE)
        if breaker and failures > BASE * FAIL_RATE:   # breaker opens on a failure surge
            added = 0
            note = "  <- breaker OPEN: retries suppressed"
        else:
            added = failures * RETRIES
            note = ""
        total_cost += load * COST_PER
        print("  {:>3}   {:>7}   {:>7}   {:>12}   ${:>6.2f}{}".format(t, load, failures, added, load * COST_PER, note))
        load = BASE + added
    print("  total cost over 4 ticks: ${:.2f}".format(total_cost))

print("WITHOUT a circuit breaker (retries amplify the load):")
run(breaker=False)
print()
print("WITH a circuit breaker + budget cap (retries suppressed on a failure surge):")
run(breaker=True)
print()
# (2) A PII leak's blast radius includes the CACHE - a cached reply is replayed.
cache_entries = 5000
tainted_serves = 1240   # times the PII-bearing cached reply was served
print("PII leak - the blast radius is not one response:")
print("  the PII-bearing reply was cached and REPLAYED {} times".format(tainted_serves))
print("  and you cannot know which of the {} entries are also tainted".format(cache_entries))
print("  -> scope the {} affected serves AND flush the whole cache, then notify legal.".format(tainted_serves))

# Output:
# WITHOUT a circuit breaker (retries amplify the load):
#   tick  attempts  failures  retries_added  cost
#     0       100        50            150   $  0.20
#     1       250       125            375   $  0.50
#     2       475       237            711   $  0.95
#     3       811       405           1215   $  1.62
#   total cost over 4 ticks: $3.27
#
# WITH a circuit breaker + budget cap (retries suppressed on a failure surge):
#   tick  attempts  failures  retries_added  cost
#     0       100        50            150   $  0.20
#     1       250       125              0   $  0.50  <- breaker OPEN: retries suppressed
#     2       100        50            150   $  0.20
#     3       250       125              0   $  0.50  <- breaker OPEN: retries suppressed
#   total cost over 4 ticks: $1.40
#
# PII leak - the blast radius is not one response:
#   the PII-bearing reply was cached and REPLAYED 1240 times
#   and you cannot know which of the 5000 entries are also tainted
#   -> scope the 1240 affected serves AND flush the whole cache, then notify legal.

**Explanation**

A two-part simulator, not a model call: the first part runs a four-tick feedback loop with and without a circuit breaker to show retries compounding load and cost, and the second part reasons about a PII leak's cache blast radius. It makes concrete why the fix for a storm is *fewer* retries (a breaker plus a budget cap), and why fixing one PII response is not enough - you must purge the whole cache.

**How the code works - step by step**
- **constants** - `BASE=100` load, `FAIL_RATE=0.5`, `RETRIES=3`, `COST_PER=0.002`.
- **`run(breaker)`** - loops 4 ticks: each tick computes failures, and either adds `failures * RETRIES` to next tick's load (no breaker) or, when the breaker is on and failures surge, suppresses the retries; accumulates cost and prints a row per tick.
- **two calls** - `run(breaker=False)` shows the amplifying storm; `run(breaker=True)` shows the breaker opening on surges and bounding the load.
- **PII block** - hard-codes `cache_entries=5000` and `tainted_serves=1240` and prints that the leaked reply was replayed and any entry could be tainted, so you scope the affected serves, flush the whole cache, and notify legal.

**In one line:** the breaker roughly halves the storm's cost, and a PII leak means purge the cache, not one reply.

**What you'll see:** Without the breaker, attempts climb 100 -> 250 -> 475 -> 811 for $3.27 total; with the breaker + budget cap, retries are suppressed on surges and the total is $1.40 (about half). Then the PII section reports the reply was replayed 1240 times across 5000 possibly-tainted entries, prescribing scope + full cache flush + notify legal.

## 7 - Write the blameless postmortem

The incident is over; now you make sure it never happens twice. Within a day of a SEV1/2 you write a **blameless postmortem**: a timeline, the two clocks, the root-cause category with a **version snapshot** (which model/prompt/index/guardrail was live), action items that fix the *system*, and a 'where we got lucky' near-miss. This block assembles one for a hallucination spike after a model version bump.

In [ ]:
# Within a day, write a BLAMELESS postmortem: timeline, the two clocks, the root-cause
# category with a VERSION SNAPSHOT, action items with owners, and "where we got lucky".
# Timeline in minutes from t0 (a provider model version bump went live at t0):
tl = {"began": 0, "detected": 240, "commander": 245, "root_cause": 260, "mitigated": 275, "resolved": 280}
mttd = tl["detected"] - tl["began"]
mttr = tl["resolved"] - tl["detected"]
duration = tl["resolved"] - tl["began"]

print("POSTMORTEM: hallucination spike after a model version bump")
print("  duration {}m   MTTD {}m   MTTR {}m".format(duration, mttd, mttr))
print()
print("  Timeline (minutes from t0):")
for k in ["began", "detected", "commander", "root_cause", "mitigated", "resolved"]:
    print("    t={:>3}  {}".format(tl[k], k.replace("_", " ")))
print()
print("  Root cause category: MODEL")
print("  Version snapshot (the crux of an AI postmortem):")
snapshot = {"model": "vendor-model v2 -> v3", "prompt": "p-1.4 (unchanged)", "index": "idx-11 (unchanged)",
            "guardrail": "g-2 (unchanged)"}
for k, v in snapshot.items():
    print("    {:<10} {}".format(k, v))
print()
print("  Action items (blameless - fix the system, not the person):")
for pri, item, owner in [("P0", "pin the exact model VERSION, not just the name", "platform"),
                          ("P0", "add numeric fact-checking to the output validator", "quality"),
                          ("P1", "alert on provider changelog / version changes", "sre")]:
    print("    {} {:<48} owner: {}".format(pri, item, owner))
print()
print("  Where we got lucky: the bad version only skewed numeric answers and was caught")
print("  before a larger launch - a near-miss that a slower MTTD would have made a headline.")

# Output:
# POSTMORTEM: hallucination spike after a model version bump
#   duration 280m   MTTD 240m   MTTR 40m
#
#   Timeline (minutes from t0):
#     t=  0  began
#     t=240  detected
#     t=245  commander
#     t=260  root cause
#     t=275  mitigated
#     t=280  resolved
#
#   Root cause category: MODEL
#   Version snapshot (the crux of an AI postmortem):
#     model      vendor-model v2 -> v3
#     prompt     p-1.4 (unchanged)
#     index      idx-11 (unchanged)
#     guardrail  g-2 (unchanged)
#
#   Action items (blameless - fix the system, not the person):
#     P0 pin the exact model VERSION, not just the name   owner: platform
#     P0 add numeric fact-checking to the output validator owner: quality
#     P1 alert on provider changelog / version changes    owner: sre
#
#   Where we got lucky: the bad version only skewed numeric answers and was caught
#   before a larger launch - a near-miss that a slower MTTD would have made a headline.

**Explanation**

A report builder, not a model call: it reconstructs a timeline from hard-coded minute marks, derives MTTD/MTTR/duration, and prints the sections a good AI postmortem needs. The load-bearing idea is the version snapshot - 'which version was running?' is almost always the crux of an AI incident - and blameless action items that pin the system, not the person.

**How the code works - step by step**
- **`tl`** - the timeline dict (began, detected, commander, root_cause, mitigated, resolved) in minutes from t0.
- **derived clocks** - `mttd`, `mttr`, and `duration` computed from the timeline marks.
- **timeline print** - iterates the milestones in order, printing each with its minute.
- **`snapshot`** - the version state (model bumped v2 -> v3, prompt/index/guardrail unchanged), the crux of the postmortem.
- **action items** - a list of (priority, fix, owner) tuples - pin the exact model version, add numeric validation, alert on provider changes - each fixing the system.
- **'where we got lucky'** - the near-miss note that a slower MTTD would have made this a headline.

**In one line:** timeline -> two clocks -> version snapshot -> system fixes -> near-miss.

**What you'll see:** A full postmortem prints: duration 280m, MTTD 240m, MTTR 40m; the ordered timeline; root-cause category MODEL with a version snapshot (vendor-model v2 -> v3, everything else unchanged); three blameless action items (P0 pin the version, P0 add numeric validation, P1 alert on provider changes) with owners; and the where-we-got-lucky near-miss.

## 8 - An illustrative on-call runbook

The last block is the 3 AM artifact itself: a minimal runbook showing the two fastest mitigations as code - a **kill switch** feature flag that degrades gracefully without a deploy, and an **auto-rollback** that shifts traffic to the last good revision when the error rate stays over the SLO.

> **Illustrative only** - it shells out to `gcloud` and calls `call_model`/`notify_oncall` helpers that are not defined here, so it is read, not run.

In [ ]:
# ON-CALL RUNBOOK - an illustrative minimal example (kill switch + auto-rollback).
import os, subprocess

# A feature flag you can flip WITHOUT a deploy - the fastest safe mitigation.
KILL_SWITCH = os.environ.get("AI_KILL_SWITCH", "off") == "on"

def answer(prompt: str) -> str:
    if KILL_SWITCH:
        # degrade gracefully: a safe non-AI fallback beats a confidently wrong AI answer
        return "Our assistant is briefly unavailable; here is our help center instead."
    return call_model(prompt)   # the normal AI path

# Auto-rollback: when the error rate stays over the SLO, shift traffic to the last good revision.
def auto_rollback(error_rate: float, threshold: float = 0.20) -> None:
    if error_rate <= threshold:
        return
    # the 12.6 traffic lever, run from the on-call machine:
    # gcloud run services update-traffic ai-svc --to-revisions LAST_GOOD=100
    subprocess.run(["gcloud", "run", "services", "update-traffic", "ai-svc",
                    "--to-revisions", "LAST_GOOD=100"], check=False)
    notify_oncall("auto-rollback fired: error rate over SLO, traffic shifted to LAST_GOOD")
# Output: (illustrative minimal example - needs gcloud + a deployed service and an on-call hook; not run here.)

**Explanation**

A reference skeleton, not a runnable cell: it sketches the two mitigation patterns you would wire into a real service. The kill switch reads an env-var flag and returns a safe non-AI fallback when flipped; the auto-rollback runs the 12.6 traffic lever via `gcloud` when the error rate breaches the SLO. It ties the lesson's levers to real on-call plumbing.

**How the code works - step by step**
- **`KILL_SWITCH`** - reads `AI_KILL_SWITCH` from the environment; on means the AI path is disabled.
- **`answer`** - if the kill switch is on, returns a graceful help-center fallback; otherwise calls `call_model` (the normal path).
- **`auto_rollback`** - returns early if the error rate is within threshold; otherwise `subprocess.run`s the `gcloud run services update-traffic` command to send 100% of traffic to `LAST_GOOD`, then notifies on-call.

**In one line:** a flag that degrades gracefully plus an SLO-triggered traffic rollback.

**What you'll see:** No output - it is an illustrative minimal example that needs `gcloud`, a deployed service, and an on-call hook, so it is shown for reference and not executed here (as the trailing comment states).

Across seven runnable blocks you built an entire AI-incident lifecycle with no cloud and no API key: an AI health check that catches the gray failures uptime is blind to, a severity classifier with the MTTD/MTTR clock, a role assigner, a what-changed triage, the five rollback levers, a retry-storm-and-PII-leak simulator, and a blameless postmortem with a version snapshot - plus an illustrative on-call runbook. The through-line is detect (golden signals, not uptime) -> classify -> command -> triage -> mitigate-before-diagnose -> learn, with the AI-specific twist at every stage. Next, Lesson 12.10 closes the module with production readiness (the maturity that makes these incidents rarer and calmer), Module 13 attacks the cost a runaway threatens, and Lesson 14.4 turns a postmortem action item into a permanent regression eval.